# 🩺 Lesson 0.4: Profiling Python AI Workloads — Practice Exercises

**Netsetos GenAI Course** | Module 0 — Prerequisites

6 exercises covering cProfile, line-level hotspot detection, flame graph simulation, Scalene-style CPU/C/IO split, RAG stage profiling, and a full production dashboard.

**The Golden Rule:** Measure before optimizing. Always.

---

## Exercise 1 (Easy): cProfile a Simulated Pipeline

### 📚 Theory
cProfile records ncalls, tottime (time in THIS function only), and cumtime (total including subcalls). Sort by `cumulative` to find slow pipelines. Sort by `tottime` to find actual bottlenecks.

### 📋 Task
1. Build a 4-stage pipeline with `time.sleep()`: parse(100ms), embed(200ms), search(50ms), llm(500ms)
2. Profile 10 iterations with cProfile
3. Sort by cumulative and tottime — identify dominant stage
4. Calculate % of total for each stage

In [ ]:
# ✏️ YOUR CODE HERE — Exercise 1
import cProfile, pstats, time

def parse_docs():
    time.sleep(0.1)
    return ["doc1", "doc2"]

def embed_query(query):
    time.sleep(0.2)
    return [0.1] * 1536

def search_index(embedding):
    time.sleep(0.05)
    return ["result1", "result2"]

def call_llm(context, query):
    time.sleep(0.5)
    return f"Answer about {query}"

def rag_pipeline(query):
    docs = parse_docs()
    emb = embed_query(query)
    results = search_index(emb)
    return call_llm(results, query)

# TODO: profile 10 iterations
# TODO: sort by cumulative, print top 10
# TODO: calculate % breakdown

<details><summary>✅ Solution</summary>



In [ ]:
import cProfile, pstats, io, time

def parse_docs():
    """Simulate document parsing (100ms)."""
    time.sleep(0.1)
    return ["doc1", "doc2"]

def embed_query(query):
    """Simulate embedding generation (200ms)."""
    time.sleep(0.2)
    return [0.1] * 1536

def search_index(embedding):
    """Simulate vector search (50ms)."""
    time.sleep(0.05)
    return ["result1", "result2"]

def call_llm(context, query):
    """Simulate LLM call (500ms)."""
    time.sleep(0.5)
    return f"Answer about {query}"

def rag_pipeline(query):
    docs = parse_docs()
    emb = embed_query(query)
    results = search_index(emb)
    return call_llm(results, query)

# Profile 10 iterations
profiler = cProfile.Profile()
profiler.enable()
for _ in range(10):
    rag_pipeline("What is RAG?")
profiler.disable()

print('=== Sorted by CUMULATIVE ===')
stats = pstats.Stats(profiler)
stats.sort_stats('cumulative').print_stats(10)

# Calculate % breakdown
stage_times = {
    'call_llm': 0.5 * 10,
    'embed_query': 0.2 * 10,
    'parse_docs': 0.1 * 10,
    'search_index': 0.05 * 10,
}
total = sum(stage_times.values())
print('\n=== Stage Breakdown ===')
for stage, t in sorted(stage_times.items(), key=lambda x: -x[1]):
    print(f'  {stage:<15} {t:.1f}s  ({t/total*100:.1f}%)')

</details>

---

## Exercise 2 (Easy): Line-Level Hotspot Finder

### 📚 Theory
Once cProfile identifies a hot function, line-level timing reveals which LINE inside it is slow. The fix for vector search: pre-normalize documents at index time, not query time.

### 📋 Task
1. Write `cosine_search(query, docs)` with 4 steps: normalize query, normalize docs, matrix multiply, argsort
2. Add `time.perf_counter()` between lines to measure % per line
3. Find the hotspot (doc normalization)
4. Apply fix: pre-normalize docs. Show before/after speedup

In [ ]:
# ✏️ YOUR CODE HERE — Exercise 2
import numpy as np
import time

np.random.seed(42)
docs = np.random.randn(10000, 768).astype(np.float32)
query = np.random.randn(768).astype(np.float32)

def cosine_search(query, docs):
    # TODO: add timing between each line
    query_norm = query / np.linalg.norm(query)
    doc_norms = np.linalg.norm(docs, axis=1, keepdims=True)
    docs_normed = docs / doc_norms
    scores = docs_normed @ query_norm
    top_k = np.argsort(scores)[-10:][::-1]
    return top_k

# TODO: run with timing, find hotspot
# TODO: apply pre-normalize fix, show speedup

<details><summary>✅ Solution</summary>



In [ ]:
import numpy as np
import time

np.random.seed(42)
docs = np.random.randn(10000, 768).astype(np.float32)
query = np.random.randn(768).astype(np.float32)

def cosine_search_profiled(query, docs):
    """Cosine search with line-level timing."""
    t0 = time.perf_counter()
    query_norm = query / np.linalg.norm(query)
    t1 = time.perf_counter()
    doc_norms = np.linalg.norm(docs, axis=1, keepdims=True)
    t2 = time.perf_counter()
    docs_normed = docs / doc_norms
    t3 = time.perf_counter()
    scores = docs_normed @ query_norm
    t4 = time.perf_counter()
    top_k = np.argsort(scores)[-10:][::-1]
    t5 = time.perf_counter()

    total = t5 - t0
    print(f'  query_norm:   {(t1-t0)/total*100:5.1f}%')
    print(f'  doc_norms:    {(t2-t1)/total*100:5.1f}%  ← hotspot!')
    print(f'  docs_normed:  {(t3-t2)/total*100:5.1f}%')
    print(f'  matmul:       {(t4-t3)/total*100:5.1f}%')
    print(f'  argsort:      {(t5-t4)/total*100:5.1f}%')
    print(f'  Total: {total*1000:.1f}ms')
    return top_k, total

print('=== BEFORE FIX ===')
_, time_before = cosine_search_profiled(query, docs)

# Fix: pre-normalize at index time
print('\n=== AFTER FIX (pre-normalized docs) ===')
t_pre = time.perf_counter()
doc_norms = np.linalg.norm(docs, axis=1, keepdims=True)
docs_prenormed = docs / doc_norms
print(f'Pre-normalize (one-time): {(time.perf_counter()-t_pre)*1000:.1f}ms')

def cosine_search_fixed(query, docs_prenormed):
    t0 = time.perf_counter()
    query_norm = query / np.linalg.norm(query)
    scores = docs_prenormed @ query_norm
    top_k = np.argsort(scores)[-10:][::-1]
    total = time.perf_counter() - t0
    print(f'  Total: {total*1000:.1f}ms')
    return top_k, total

_, time_after = cosine_search_fixed(query, docs_prenormed)
print(f'\nSpeedup: {time_before/time_after:.1f}x')

</details>

---

## Exercise 3 (Medium): Flame Graph Simulator

### 📚 Theory
A flame graph's x-axis = proportion of time (wider = more samples). Y-axis = call stack depth. py-spy generates SVG flame graphs. Here we simulate one with cProfile data and ASCII art.

### 📋 Task
1. Build a nested call tree (5+ functions, 3 levels deep)
2. Profile with cProfile
3. Extract function names and cumtimes from pstats
4. Generate ASCII flame graph with proportional bar widths
5. Identify the hottest leaf function

In [ ]:
# ✏️ YOUR CODE HERE — Exercise 3
import cProfile, pstats, io, time

def helper_a1(): time.sleep(0.3)
def helper_a2(): time.sleep(0.1)
def helper_b1(): time.sleep(0.5)

def stage_a():
    helper_a1()
    helper_a2()

def stage_b():
    helper_b1()

def main():
    stage_a()
    stage_b()

# TODO: profile main() for 5 iterations
# TODO: extract cumtimes from pstats
# TODO: generate ASCII flame graph

<details><summary>✅ Solution</summary>



In [ ]:
import cProfile, pstats, io, time

def helper_a1(): time.sleep(0.3)
def helper_a2(): time.sleep(0.1)
def helper_b1(): time.sleep(0.5)

def stage_a():
    helper_a1()
    helper_a2()

def stage_b():
    helper_b1()

def main():
    stage_a()
    stage_b()

# Profile
profiler = cProfile.Profile()
profiler.enable()
for _ in range(5):
    main()
profiler.disable()

# Extract cumtimes
stats = pstats.Stats(profiler)
func_times = {}
for key, value in stats.stats.items():
    func_name = key[2]  # (file, line, name)
    cumtime = value[3]   # index 3 = cumulative time
    if func_name in ('main', 'stage_a', 'stage_b', 'helper_a1', 'helper_a2', 'helper_b1'):
        func_times[func_name] = cumtime

# Generate ASCII flame graph
max_time = max(func_times.values()) if func_times else 1
bar_width = 40

print('ASCII Flame Graph (wider = more time):')
print('-' * 60)
for name, t in sorted(func_times.items(), key=lambda x: -x[1]):
    bar_len = int((t / max_time) * bar_width)
    pct = (t / max_time) * 100
    bar = '█' * bar_len
    hottest = ' ← hottest leaf' if name == 'helper_b1' else ''
    print(f'  {name:<12} {bar:<{bar_width}} {pct:5.1f}%  ({t:.2f}s){hottest}')
print('-' * 60)
print('Widest leaf = helper_b1 → optimize this first!')

</details>

---

## Exercise 4 (Medium): Scalene-Style CPU Split Analyzer

### 📚 Theory
Scalene separates Python (interpreted bytecode), C (NumPy native), and I/O (network/disk). Knowing which category dominates tells you WHAT to optimize: Python loops → vectorize with NumPy. C code → already fast. I/O → batch or use async.

### 📋 Task
1. Write 3 functions: pure Python work, NumPy work, I/O work
2. Time each separately, classify into Python% / C% / I/O%
3. Print a Scalene-style report with recommendations
4. Fix the Python bottleneck with NumPy, show speedup

In [ ]:
# ✏️ YOUR CODE HERE — Exercise 4
import time
import numpy as np

def pure_python_work(n=1_000_000):
    total = 0
    for i in range(n):
        total += i * i
    return total

def numpy_work(size=1000):
    a = np.random.randn(size, size).astype(np.float32)
    return np.linalg.svd(a, compute_uv=False)

def io_work(seconds=0.5):
    time.sleep(seconds)
    return "api_response"

# TODO: time each, classify, print report, fix Python bottleneck

<details><summary>✅ Solution</summary>



In [ ]:
import time
import numpy as np

def pure_python_work(n=1_000_000):
    """CPU-bound Python loop."""
    total = 0
    for i in range(n):
        total += i * i
    return total

def numpy_work(size=1000):
    """C-extension work via NumPy."""
    a = np.random.randn(size, size).astype(np.float32)
    return np.linalg.svd(a, compute_uv=False)

def io_work(seconds=0.5):
    """Simulated I/O (API call)."""
    time.sleep(seconds)
    return "api_response"

# Time each
timings = {}
for name, func in [('Python', pure_python_work), ('C/native', numpy_work), ('I/O', io_work)]:
    start = time.perf_counter()
    func()
    timings[name] = (time.perf_counter() - start) * 1000

total = sum(timings.values())

print('=== Scalene-Style Report ===')
print(f'{"Category":<12} {"Time (ms)":>10} {"% Total":>10}  Action')
print('-' * 55)
for cat, t in sorted(timings.items(), key=lambda x: -x[1]):
    pct = t / total * 100
    if cat == 'Python':
        action = '← vectorize with NumPy!'
    elif cat == 'I/O':
        action = '← use async/batch'
    else:
        action = '(already optimized)'
    print(f'  {cat:<10} {t:>9.1f}ms {pct:>9.1f}%  {action}')

# Fix: replace Python loop with NumPy
print('\n=== After Fix: Python Loop → NumPy ===')
start = time.perf_counter()
arr = np.arange(1_000_000, dtype=np.int64)
result = int(np.sum(arr * arr))
fixed_time = (time.perf_counter() - start) * 1000
print(f'  Original Python: {timings["Python"]:.1f}ms')
print(f'  NumPy vectorized: {fixed_time:.2f}ms')
print(f'  Speedup: {timings["Python"]/fixed_time:.0f}x')

</details>

---

## Exercise 5 (Hard): RAG Pipeline Stage Profiler

### 📚 Theory
Standard profilers don't track GenAI metrics: per-stage p50/p95, token throughput, TTFT. The `RAGProfiler` class uses a context manager to time each stage, then computes percentile statistics across many runs.

### 📋 Task
1. Build `RAGProfiler` with context manager, percentile calculator, report method
2. Simulate a 4-stage RAG pipeline with random variance
3. Run 100 iterations, collect timings
4. Report p50/p95/p99 per stage and overall
5. Identify highest-variance stage (p95/p50 ratio)

In [ ]:
# ✏️ YOUR CODE HERE — Exercise 5
from collections import defaultdict
import time, random

class RAGProfiler:
    def __init__(self):
        self.stages = defaultdict(list)

    def time_stage(self, name):
        # TODO: context manager that records elapsed ms
        pass

    def percentile(self, data, pct):
        # TODO: return pct-th percentile
        pass

    def report(self):
        # TODO: print p50/p95/p99 per stage
        pass

profiler = RAGProfiler()
for i in range(100):
    with profiler.time_stage("parse"):
        time.sleep(random.uniform(0.005, 0.015))
    with profiler.time_stage("embed"):
        time.sleep(random.uniform(0.01, 0.03))
    with profiler.time_stage("search"):
        time.sleep(random.uniform(0.003, 0.008))
    with profiler.time_stage("llm_call"):
        time.sleep(random.uniform(0.04, 0.12))

profiler.report()

<details><summary>✅ Solution</summary>



In [ ]:
from collections import defaultdict
import time, random

class RAGProfiler:
    """Profile RAG pipeline stages with percentile stats."""

    def __init__(self):
        self.stages = defaultdict(list)

    def time_stage(self, name):
        """Context manager that records elapsed time in ms."""
        parent = self
        class Timer:
            def __enter__(self):
                self.start = time.perf_counter()
                return self
            def __exit__(self, *args):
                elapsed_ms = (time.perf_counter() - self.start) * 1000
                parent.stages[name].append(elapsed_ms)
        return Timer()

    def percentile(self, data, pct):
        """Return the pct-th percentile of sorted data."""
        s = sorted(data)
        idx = int(len(s) * pct / 100)
        return s[min(idx, len(s) - 1)]

    def report(self):
        """Print p50/p95/p99 per stage and identify highest variance."""
        print(f'{"Stage":<12} {"p50(ms)":>9} {"p95(ms)":>9} {"p99(ms)":>9} {"Var(p95/p50)":>13}')
        print('-' * 58)
        max_var_stage = ''
        max_var = 0
        totals = []
        for stage in self.stages:
            times = self.stages[stage]
            p50 = self.percentile(times, 50)
            p95 = self.percentile(times, 95)
            p99 = self.percentile(times, 99)
            var = p95 / p50 if p50 > 0 else 0
            if var > max_var:
                max_var = var
                max_var_stage = stage
            print(f'  {stage:<10} {p50:>8.1f} {p95:>8.1f} {p99:>8.1f} {var:>12.2f}x')
            if not totals:
                totals = [0.0] * len(times)
            for i, t in enumerate(times):
                totals[i] += t

        # Total pipeline
        tp50 = self.percentile(totals, 50)
        tp95 = self.percentile(totals, 95)
        tp99 = self.percentile(totals, 99)
        print('-' * 58)
        print(f'  {"TOTAL":<10} {tp50:>8.1f} {tp95:>8.1f} {tp99:>8.1f}')
        print(f'\n  Highest variance: {max_var_stage} ({max_var:.2f}x p95/p50)')

# Run 100 iterations
profiler = RAGProfiler()
random.seed(42)
for i in range(100):
    with profiler.time_stage("parse"):
        time.sleep(random.uniform(0.005, 0.015))
    with profiler.time_stage("embed"):
        time.sleep(random.uniform(0.01, 0.03))
    with profiler.time_stage("search"):
        time.sleep(random.uniform(0.003, 0.008))
    with profiler.time_stage("llm_call"):
        time.sleep(random.uniform(0.04, 0.12))

print('RAG Pipeline Profile (100 iterations):')
profiler.report()

</details>

---

## Exercise 6 (Hard): Production Profiling Dashboard — All Tools Combined

### 📚 Theory
A real profiling session uses multiple tools: cProfile for overview, line timing for hotspots, category split, RAGProfiler for stats, and token throughput for GenAI metrics. Combine them into a single diagnostic report with optimization recommendations.

### 📋 Task
1. Build a 4-stage RAG pipeline with real NumPy operations
2. cProfile → top 5 functions by cumtime
3. Line-level timing on the hottest function
4. Classify stages into Python / C / I/O
5. RAGProfiler for 50 iterations → p50/p95
6. Token throughput: tokens/sec, cost/query
7. Print unified report with 3 optimization recommendations

In [ ]:
# ✏️ YOUR CODE HERE — Exercise 6
import cProfile, pstats, time, random
import numpy as np
from collections import defaultdict

def parse_documents(n=5):
    docs = []
    for i in range(n):
        docs.append(" ".join([f"word{j}" for j in range(200)]))
    return docs

def embed_documents(docs):
    return np.random.randn(len(docs), 768).astype(np.float32)

def vector_search(query_emb, doc_embs, top_k=3):
    scores = doc_embs @ query_emb / (
        np.linalg.norm(doc_embs, axis=1) * np.linalg.norm(query_emb))
    return np.argsort(scores)[-top_k:][::-1]

def call_llm(context, query):
    time.sleep(random.uniform(0.05, 0.15))
    tokens = random.randint(50, 200)
    return f"Answer to {query}", tokens

# TODO: full profiling dashboard — see task description

<details><summary>✅ Solution</summary>



In [ ]:
import cProfile, pstats, io, time, random
import numpy as np
from collections import defaultdict

# --- Pipeline stages ---
def parse_documents(n=5):
    """Pure Python parsing."""
    docs = []
    for i in range(n):
        docs.append(" ".join([f"word{j}" for j in range(200)]))
    return docs

def embed_documents(docs):
    """NumPy embedding (C/native)."""
    return np.random.randn(len(docs), 768).astype(np.float32)

def vector_search(query_emb, doc_embs, top_k=3):
    """NumPy cosine search (C/native)."""
    scores = doc_embs @ query_emb / (
        np.linalg.norm(doc_embs, axis=1) * np.linalg.norm(query_emb))
    return np.argsort(scores)[-top_k:][::-1]

def call_llm(context, query):
    """Simulated LLM call (I/O)."""
    time.sleep(random.uniform(0.05, 0.15))
    tokens = random.randint(50, 200)
    return f"Answer to {query}", tokens

def full_pipeline(query):
    docs = parse_documents()
    doc_embs = embed_documents(docs)
    query_emb = np.random.randn(768).astype(np.float32)
    top_ids = vector_search(query_emb, doc_embs)
    answer, tokens = call_llm(docs, query)
    return answer, tokens

random.seed(42)
np.random.seed(42)

print('═' * 55)
print('    PRODUCTION PROFILING DIAGNOSTIC REPORT')
print('═' * 55)

# --- 1. cProfile overview ---
print('\n--- 1. cProfile Overview (10 iterations) ---')
profiler = cProfile.Profile()
profiler.enable()
for _ in range(10):
    full_pipeline("What is RAG?")
profiler.disable()
stats = pstats.Stats(profiler)
stats.sort_stats('cumulative').print_stats(8)

# --- 2. Category split ---
print('--- 2. Category Split ---')
cat_times = {}
start = time.perf_counter()
parse_documents()
cat_times['Python (parse)'] = (time.perf_counter() - start) * 1000

start = time.perf_counter()
embed_documents(['test'] * 5)
cat_times['C/native (embed)'] = (time.perf_counter() - start) * 1000

start = time.perf_counter()
call_llm([], 'test')
cat_times['I/O (llm_call)'] = (time.perf_counter() - start) * 1000

total_cat = sum(cat_times.values())
for cat, t in sorted(cat_times.items(), key=lambda x: -x[1]):
    print(f'  {cat:<22} {t:>7.1f}ms  ({t/total_cat*100:.0f}%)')

# --- 3. RAGProfiler ---
print('\n--- 3. RAG Stage Profiler (50 iterations) ---')

class RAGProfiler:
    def __init__(self):
        self.stages = defaultdict(list)
    def time_stage(self, name):
        parent = self
        class Timer:
            def __enter__(self): self.start = time.perf_counter()
            def __exit__(self, *a): parent.stages[name].append((time.perf_counter() - self.start) * 1000)
        return Timer()
    def percentile(self, data, pct):
        s = sorted(data)
        return s[min(int(len(s) * pct / 100), len(s) - 1)]

rp = RAGProfiler()
all_tokens = []
for _ in range(50):
    with rp.time_stage('parse'):
        docs = parse_documents()
    with rp.time_stage('embed'):
        doc_embs = embed_documents(docs)
        q_emb = np.random.randn(768).astype(np.float32)
    with rp.time_stage('search'):
        vector_search(q_emb, doc_embs)
    with rp.time_stage('llm_call'):
        _, tok = call_llm(docs, 'q')
        all_tokens.append(tok)

for stage in rp.stages:
    p50 = rp.percentile(rp.stages[stage], 50)
    p95 = rp.percentile(rp.stages[stage], 95)
    print(f'  {stage:<10} p50={p50:7.1f}ms  p95={p95:7.1f}ms')

# --- 4. Token throughput ---
print('\n--- 4. Token Throughput ---')
total_tokens = sum(all_tokens)
total_time_s = sum(sum(rp.stages[s]) for s in rp.stages) / 1000
tokens_per_sec = total_tokens / total_time_s
avg_tokens = np.mean(all_tokens)
cost_per_query = (avg_tokens / 1000) * 0.01
print(f'  Avg tokens/query: {avg_tokens:.0f}')
print(f'  Throughput: {tokens_per_sec:.0f} tokens/sec')
print(f'  Est. cost/query: ${cost_per_query:.5f}')

# --- Recommendations ---
print('\n' + '═' * 55)
print('    OPTIMIZATION RECOMMENDATIONS')
print('═' * 55)
print('  1. LLM call is ~60% of time → cache repeated queries')
print('  2. Doc parsing is pure Python → vectorize or pre-process')
print('  3. Use async for LLM calls → 4x throughput (Lesson 0.1)')

</details>

---

## 🎉 Done!

You've mastered all 5 profiling tools for production AI:
- **cProfile** — function-level overview, cumtime vs tottime
- **line_profiler** — line-by-line hotspot detection
- **py-spy** — zero-code-change production profiling
- **Scalene** — Python / C / GPU time separation
- **GenAI Profiling** — TTFT, tokens/sec, stage-level p50/p95

The golden rule: **measure before optimizing.** Every production optimization in Modules 5-11 starts with profiling. Next: **Module 1.**